In [ ]:
# train.ipynb
import os
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import joblib
from scipy.signal import welch, spectrogram
from scipy.stats import entropy
from nolds import lyap_r
from mne.preprocessing import ICA
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.ensemble import BalancedRandomForestClassifier
from joblib import Parallel, delayed

mne.set_log_level("WARNING")

# ── Set DATA_DIR to the folder containing your .edf files ─────────────────
DATA_DIR = r"C:\Users\aaww8\OneDrive\바탕 화면\phython workspace\.vscode\chb-mit-scalp-eeg-database-1.0.0"
# GitHub: DATA_DIR = "data"
# ──────────────────────────────────────────────────────────────────────────

# ── 1. Load & preprocess ───────────────────────────────────────────────────
edf_file = os.path.join(DATA_DIR, "chb01_03.edf")
raw      = mne.io.read_raw_edf(edf_file, preload=True, verbose=False)
raw_original = raw.copy()

channels = [
    "FP1-F7", "F7-T7",   "T7-P7",   "P7-O1",
    "FP1-F3", "F3-C3",   "C3-P3",   "P3-O1",
    "FP2-F4", "F4-C4",   "C4-P4",   "P4-O2",
    "FP2-F8", "F8-T8",   "T8-P8-1", "T8-P8-0",
    "P8-O2",  "FZ-CZ",   "CZ-PZ"
]
raw.pick_channels(channels)
raw.filter(l_freq=0.5, h_freq=40, verbose=False)  # bandpass: keep 0.5–40 Hz
raw.resample(sfreq=128, verbose=False)             # 256 Hz → 128 Hz (lossless above 40 Hz)

print("1/3  Before vs After preprocessing (filter + resample)")
raw_original.plot(duration=2, scalings=dict(eeg=20e-6), title="Before Preprocessing (Original)")
raw.plot(duration=2, scalings=dict(eeg=20e-6),          title="After Preprocessing (Filtered & Resampled)")

# Average re-referencing — remove common-mode noise across electrodes
raw.set_eeg_reference("average", projection=True, verbose=False)
raw.apply_proj()

# ICA — decompose into 15 independent components
ica = ICA(n_components=15, max_iter="auto", random_state=97)
ica.fit(raw, verbose=False)

print("2/3  ICA components — identify artifact (eye-blink typically ICA000)")
ica.plot_sources(raw)

ica.exclude = [0]  # ICA000: ocular artifact (eye-blink)
print("3/3  ICA overlay — red: original, black: ICA000 removed")
ica.plot_overlay(raw, exclude=[0])

raw_clean = raw.copy()
ica.apply(raw_clean, verbose=False)

# ── 2. Epoching ────────────────────────────────────────────────────────────
epochs     = mne.make_fixed_length_epochs(raw_clean, duration=2.0, preload=True, verbose=False)
data_array = epochs.get_data()
print(f"data_array shape: {data_array.shape}")  # (1800, 19, 256)

# ── 3. Time-domain features ────────────────────────────────────────────────
# axis=2: compute along the time axis within each epoch
feature_ptp = np.ptp(data_array, axis=2)  # Peak-to-Peak: spike amplitude indicator
feature_var = np.var(data_array, axis=2)  # Variance: signal spread

print(f"Peak-to-Peak shape: {feature_ptp.shape}")  # (1800, 19)
print(f"Variance shape:     {feature_var.shape}")  # (1800, 19)

# ── 4. Pseudo-labeling ─────────────────────────────────────────────────────
# No physician labels available for training data.
# Label epoch as spike (1) if PtP AND Variance both exceed mean+3σ on any channel.
mean_ptp      = np.mean(feature_ptp, axis=0)
std_ptp       = np.std(feature_ptp,  axis=0)
threshold_ptp = mean_ptp + (3 * std_ptp)

mean_var      = np.mean(feature_var, axis=0)
std_var       = np.std(feature_var,  axis=0)
threshold_var = mean_var + (3 * std_var)

is_spike_both         = (feature_ptp > threshold_ptp) & (feature_var > threshold_var)
suspected_epochs_both = np.where(np.any(is_spike_both, axis=1))[0]

print(f"Suspected spike epochs: {len(suspected_epochs_both)}")
print(f"Epoch indices: {suspected_epochs_both}")

if len(suspected_epochs_both) > 0:
    first_suspect = suspected_epochs_both[0]
    start_time    = first_suspect * 2.0
    print(f"Suspected spike epoch — Epoch {first_suspect} ({start_time}–{start_time+2}s), flagged by PtP AND Variance > mean+3σ")
    raw_clean.plot(
        start=start_time, duration=2.0,
        scalings=dict(eeg=20e-6),
        title=f"Suspected Spike — Epoch {first_suspect} ({start_time}s)"
    )

# ── 5. Frequency-domain features (Welch PSD) ──────────────────────────────
# Welch method: averaged FFT over overlapping segments → stable PSD estimate
freqs, psd  = welch(data_array, fs=128, nperseg=256, axis=-1)
delta_power = np.mean(psd[:, :, (freqs >= 0.5) & (freqs < 4)],  axis=-1)
theta_power = np.mean(psd[:, :, (freqs >= 4)   & (freqs < 8)],  axis=-1)
alpha_power = np.mean(psd[:, :, (freqs >= 8)   & (freqs < 12)], axis=-1)
beta_power  = np.mean(psd[:, :, (freqs >= 12)  & (freqs < 30)], axis=-1)

# ── 6. Spectral entropy ──────────────────────────────────────
# Measures frequency distribution disorder — high entropy = uniform spread
n_epochs, n_channels, n_times = data_array.shape

def spectral_entropy(signal, sf, nperseg=256, normalize=True):
    _, psd_s = welch(signal, sf, nperseg=nperseg)
    if normalize:
        psd_s = psd_s / np.sum(psd_s)
    return entropy(psd_s, base=2)

spectral_entropy_features = np.apply_along_axis(
    spectral_entropy, 2, data_array, sf=128  
)

# ── 7. Lyapunov exponent (parallel) ───────────────────────────────────────
# Measures signal unpredictability — seizures increase nonlinear dynamics
def lyapunov_exponent(signal, emb_dim=10, lag=None):
    if lag is None:
        lag = emb_dim // 2
    return lyap_r(signal, emb_dim=emb_dim, lag=lag)

signals  = data_array.reshape(-1, n_times)
lya_flat = Parallel(n_jobs=-1)(delayed(lyapunov_exponent)(s) for s in signals)
lyapunov_features = np.array(lya_flat).reshape(n_epochs, n_channels)

# ── 8. Time-frequency features (Spectrogram) ──────────────────────────────
# Tracks how band power changes over time within each epoch
freqs_spec, _, spec = spectrogram(data_array, fs=128, nperseg=64, axis=-1)
delta_power_spec_mean = np.mean(np.mean(spec[:, :, (freqs_spec >= 0.5) & (freqs_spec < 4)],  axis=-1), axis=2)
theta_power_spec_mean = np.mean(np.mean(spec[:, :, (freqs_spec >= 4)   & (freqs_spec < 8)],  axis=-1), axis=2)
alpha_power_spec_mean = np.mean(np.mean(spec[:, :, (freqs_spec >= 8)   & (freqs_spec < 12)], axis=-1), axis=2)
beta_power_spec_mean  = np.mean(np.mean(spec[:, :, (freqs_spec >= 12)  & (freqs_spec < 30)], axis=-1), axis=2)

# ── 9. Concatenate all features ───────────────────────────────────────────
features = np.concatenate((
    feature_ptp, feature_var,
    delta_power, theta_power, alpha_power, beta_power,
    delta_power_spec_mean, theta_power_spec_mean, alpha_power_spec_mean, beta_power_spec_mean,
    spectral_entropy_features, lyapunov_features
), axis=1)
print(f"features shape: {features.shape}")  # (1800, 228)

labels = np.zeros(len(data_array))
labels[suspected_epochs_both] = 1

# ── 10. Train / test split ─────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# ── 11. Balanced Random Forest ────────────────────────────────────────────
# Resampling-based approach to handle class imbalance (few spike epochs)
brf = BalancedRandomForestClassifier(n_estimators=100, random_state=42)
brf.fit(X_train, y_train)
y_pred_brf = brf.predict(X_test)

print("\nBalanced Random Forest:")
print(classification_report(y_test, y_pred_brf))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_brf))

# ── 12. Feature importance → select top 30 ────────────────────────────────
importances    = brf.feature_importances_
feature_names  = [f"Feature_{i}" for i in range(X_train.shape[1])]
feature_imp_df = pd.DataFrame({
    "Feature":    feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

print("\nTop 20 feature importances (BRF) — used to select final 30 features for XGBoost")
plt.figure(figsize=(10, 8))
sns.barplot(x="Importance", y="Feature", data=feature_imp_df.head(20), palette="viridis")
plt.title("Top 20 Feature Importances (Spike Detection)")
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

top_features_idx = feature_imp_df.index[:30]
X_train_selected = X_train[:, top_features_idx]
X_test_selected  = X_test[:,  top_features_idx]
print(f"After feature selection: {X_train_selected.shape}")

# ── 13. XGBoost ───────────────────────────────────────────────────────────
# scale_pos_weight: penalizes missed spikes proportional to class imbalance
num_negative = np.sum(y_train == 0)
num_positive = np.sum(y_train == 1)
weight_ratio = num_negative / num_positive
print(f"\nscale_pos_weight: {weight_ratio:.2f}")

xgb_model = xgb.XGBClassifier(
    scale_pos_weight=weight_ratio,
    n_estimators=100,
    random_state=42,
    eval_metric="logloss"
)
xgb_model.fit(X_train_selected, y_train)

# Threshold comparison: 50% / 30% / 15%
y_pred_proba = xgb_model.predict_proba(X_test_selected)[:, 1]
for thresh in [0.50, 0.30, 0.15]:
    y_pred_t = (y_pred_proba >= thresh).astype(int)
    print(f"\n[Threshold {int(thresh*100)}%]")
    print(classification_report(y_test, y_pred_t))
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_t))

# ── 14. Save model and feature indices ────────────────────────────────────
joblib.dump(xgb_model,        "xgb_spike_detector_15percent.pkl")
joblib.dump(top_features_idx, "top_30_features_indices.pkl")
print("\nSaved: xgb_spike_detector_15percent.pkl, top_30_features_indices.pkl")